# 🚀 Advanced Views and Streaming

This notebook demonstrates two powerful Iceberg features in Trino:

1.  **Incremental Aggregations**: Since [Iceberg](https://docs.google.com/document/d/1UnhldHhe3Grz8JBngwXPA6ZZord1xMedY5ukEhZYF-A/edit?tab=t.4lz4t8og6hk2) do not yet support `MATERIALIZED VIEW`, we demonstrate how to implement manual incremental refresh partitions using `MERGE INTO`.
2.  **Change Data Capture (CDC) via `table_changes`**: Stream a ledger of exact row-level `INSERT`, `UPDATE`, and `DELETE` events between snapshots without needing a complex streaming infrastructure like Kafka.

**Prerequisites**: Make sure you have run the `setup.ipynb` notebook to configure the `iceberg` catalog.

---
## ⚙️ Connect to Trino

In [ ]:
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="default",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

## 🏗️ Part 1: Incremental Aggregations (Manual Materialized Views)

While Trino supports `MATERIALIZED VIEW` commands, the **Iceberg REST Catalog** currently does not support storing materialized view definitions (as of early 2026). 

Instead, lets implement the same logic manually:
1. Create a physical **aggregate table** (e.g., `daily_sales`).
2. Periodically run a `MERGE INTO` pipeline to **incrementally upsert** the newly aggregated data, rather than dropping and recomputing the whole table.

### 1. Create Base and Target Tables
Let's create the base `orders` table and our physical aggregate table `daily_sales`.

In [ ]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.silver.orders (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    amount DOUBLE
) WITH (format = 'PARQUET')
""")

run_query("""
CREATE TABLE IF NOT EXISTS iceberg.gold.daily_sales (
    order_date DATE,
    total_orders BIGINT,
    total_revenue DOUBLE
) WITH (format = 'PARQUET')
""")
print("✅ Base 'orders' and aggregate 'daily_sales' tables created.")

### 2. Insert Initial Data and Run First Aggregation
We will load orders for `2026-03-01` and aggregate them.

In [ ]:
run_query("""
INSERT INTO iceberg.silver.orders VALUES
    (1, DATE '2026-03-01', 101, 250.00),
    (2, DATE '2026-03-01', 102, 120.50),
    (3, DATE '2026-03-01', 103, 330.00)
""")

# Incremental MERGE pipeline (Idempotent)
run_query("""
MERGE INTO iceberg.gold.daily_sales AS tgt
USING (
    SELECT order_date, COUNT(order_id) AS total_orders, SUM(amount) AS total_revenue
    FROM iceberg.silver.orders
    -- Normally you'd filter here: WHERE order_date >= CURRENT_DATE - INTERVAL '1' DAY
    GROUP BY order_date
) AS src
ON tgt.order_date = src.order_date
WHEN MATCHED THEN 
    UPDATE SET total_orders = src.total_orders, total_revenue = src.total_revenue
WHEN NOT MATCHED THEN 
    INSERT (order_date, total_orders, total_revenue)
    VALUES (src.order_date, src.total_orders, src.total_revenue)
""")
print("✅ Initial dataset loaded and aggregated.")

print("\n📊 Querying 'daily_sales' (Initial State):")
run_query("SELECT * FROM iceberg.gold.daily_sales ORDER BY order_date")

### 3. Add New Data and Perform Incremental Refresh
Now let's add data for the next day (`2026-03-02`), and manually refresh.

Instead of hardcoding a date in our pipeline, we use the **High-Water Mark Pattern**. 
This dynamically finds the latest date already processed in `daily_sales`, and only reads source data from that date onwards.

In [ ]:
run_query("""
INSERT INTO iceberg.silver.orders VALUES
    (4, DATE '2026-03-02', 104, 500.00),
    (5, DATE '2026-03-02', 105, 150.00)
""")

print("🔄 Triggering Incremental Refresh via MERGE...")
run_query("""
MERGE INTO iceberg.gold.daily_sales AS tgt
USING (
    SELECT order_date, COUNT(order_id) AS total_orders, SUM(amount) AS total_revenue
    FROM iceberg.silver.orders
    -- Dynamically process only the latest days (High-water mark pattern)
    WHERE order_date >= (
        SELECT COALESCE(MAX(order_date), DATE '1970-01-01') 
        FROM iceberg.gold.daily_sales
    )
    GROUP BY order_date
) AS src
ON tgt.order_date = src.order_date
WHEN MATCHED THEN 
    UPDATE SET total_orders = src.total_orders, total_revenue = src.total_revenue
WHEN NOT MATCHED THEN 
    INSERT (order_date, total_orders, total_revenue)
    VALUES (src.order_date, src.total_orders, src.total_revenue)
""")
print("✅ Refresh complete.")

print("\n📊 Querying 'daily_sales' AFTER refresh:")
run_query("SELECT * FROM iceberg.gold.daily_sales ORDER BY order_date")

---


---
## 🎥 Part 2: Change Data Capture (CDC) via `table_changes`

A dashboard needs the final, aggregated state (like the view above). But an event-driven team wants a live stream:
_"Give us every row that changed in the customer table in the last hour."_

Iceberg is uniquely positioned for this. Through its snapshot lineage, Trino provides a built-in table function
`table_changes` that calculates the precise diff between two snapshots.
No Kafka, Debezium, or tricky timestamp tracking involved!

> **⚠️ Trino limitation (as of 2026):** `table_changes` only supports **append-only** (INSERT) snapshots.
> Tables that have been modified via `UPDATE` or `DELETE` through Trino produce **positional delete files**
> (Iceberg V2 merge-on-read). The `table_changes` function cannot diff those snapshots and raises
> `NOT_SUPPORTED`. Neither `format_version=1` nor `write_delete_mode='copy-on-write'` are accepted
> by the Polaris REST catalog. The only working approach is an **insert-only event-log table**.


### 1. Setup the Customer Profile Table
First, we define our table and put some baseline `INSERT` data.

In [ ]:
# WHY this table is plain Iceberg V2 with no special write mode settings:
#
# Trino's table_changes function only supports append-only (INSERT) snapshots.
# Any UPDATE or DELETE on an Iceberg V2 table via Trino writes positional delete
# files (merge-on-read). table_changes cannot read those files and raises:
#   TrinoUserError: Table uses features which are not yet supported by the table_changes function
#
# Attempted workarounds that did NOT work with Polaris REST catalog:
#   ✗  format_version = 1            → rejected by Polaris (not supported)
#   ✗  write_delete_mode = 'copy-on-write' → rejected by Polaris (not supported)
#   ✗  write_update_mode = 'copy-on-write' → rejected by Polaris (not supported)
#
# Working solution: model CDC as an INSERT-only event log.
# Instead of mutating rows, every state change is appended as a new event row.
# table_changes then surfaces each INSERT as a '_change_type = insert' event,
# giving consumers the full ordered history of what changed and when.
run_query("""
CREATE OR REPLACE TABLE iceberg.silver.customers (
    id BIGINT,
    name VARCHAR,
    status VARCHAR
)
""")

run_query("""
INSERT INTO iceberg.silver.customers VALUES
    (1, 'Alice', 'ACTIVE'),
    (2, 'Bob', 'ACTIVE'),
    (3, 'Charlie', 'ACTIVE')
""")

print("📋 Baseline Customer Profile Table:")
run_query("SELECT * FROM iceberg.silver.customers ORDER BY id")

### 2. Capture the Baseline Snapshot ID
We need to know where our observation “starts”. We’ll grab the current snapshot of the table. Anything happening *after* this ID will be our CDC stream.

In [ ]:
rows = run_query("""
SELECT snapshot_id 
FROM iceberg.silver."customers$snapshots" 
ORDER BY committed_at DESC LIMIT 1
""", display=False)

baseline_snapshot_id = rows[0][0]
print(f"📌 Logged baseline snapshot ID: {baseline_snapshot_id}")

### 3. Simulate System Activity (Insert-Only CDC Events)

We append new events to represent state changes — **no `UPDATE` or `DELETE`** is used.

This is intentional: `table_changes` in Trino only works on snapshots created by `INSERT`.
When Trino executes `UPDATE` or `DELETE`, it writes **positional delete files** to the data
layer (Iceberg V2 merge-on-read). The `table_changes` function cannot interpret those files
and raises a `NOT_SUPPORTED` error. To work around this, we model every state transition
as a new INSERT row — exactly like an event-sourcing or outbox pattern:

| What you'd normally do | What we do instead |
|------------------------|--------------------|
| `UPDATE customers SET status='INACTIVE' WHERE id=1` | `INSERT` a new row `(1, 'Alice', 'INACTIVE')` |
| `DELETE FROM customers WHERE id=2` | `INSERT` a tombstone row `(2, 'Bob', 'DELETED')` |
| `INSERT INTO customers ...` | `INSERT` as normal |

Downstream consumers reconstruct the current state by taking the **latest row per `id`**.


In [ ]:
# No UPDATE or DELETE — every state change is modelled as a new INSERT event.
# Consumers read the latest row per id to get the current state.
run_query("""
INSERT INTO iceberg.silver.customers VALUES
    (1, 'Alice', 'INACTIVE'),
    (2, 'Bob',   'DELETED'),
    (4, 'Dave',  'ACTIVE')
""")

print("📋 Full event log (all rows, including superseded states):")
run_query("SELECT * FROM iceberg.silver.customers ORDER BY id, status")

print("\n📋 Current state (latest event per customer):")
run_query("""
SELECT id, name, status
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _change_version_id DESC) AS rn
    FROM TABLE(iceberg.system.table_changes(
        schema_name => 'silver',
        table_name  => 'customers',
        start_snapshot_id => 0,
        end_snapshot_id   => (SELECT MAX(snapshot_id) FROM iceberg.silver."customers$snapshots")
    ))
) WHERE rn = 1
ORDER BY id
""")

### 4. Query `table_changes` for the CDC Stream 🌊
Now, the downstream team wants the event log. We use the `system.table_changes` function built into Trino Iceberg connector.

**Syntax:**
`SELECT * FROM TABLE(system.table_changes('schema', 'table', start_snapshot_id, end_snapshot_id))`


In [ ]:
print("📥 CDC Stream (Diff logically emitted since baseline):\n")

# Get the latest snapshot ID to use as the end bound
rows = run_query("""
SELECT snapshot_id 
FROM iceberg.silver.\"customers$snapshots\" 
ORDER BY committed_at DESC LIMIT 1
""", display=False)
current_snapshot_id = rows[0][0]

query = f"""
SELECT
    _change_type,
    id,
    name,
    status,
    _change_version_id
FROM TABLE(iceberg.system.table_changes(
    schema_name => 'silver',
    table_name => 'customers',
    start_snapshot_id => {baseline_snapshot_id},
    end_snapshot_id => {current_snapshot_id}
))
ORDER BY _change_version_id, id
"""
run_query(query)

**Understanding the Output:**

Because `table_changes` only supports append-only snapshots (Trino limitation), every row
in the output will have `_change_type = 'insert'`.

- **`_change_version_id`** — the snapshot ID in which this row was written. Rows from the
  baseline INSERT share one snapshot ID; the mutation batch shares another.
- **Reconstructing current state** — query `table_changes` for all history, then take the
  row with the highest `_change_version_id` per `id`. The `status` column tells you the
  latest known state (`'ACTIVE'`, `'INACTIVE'`, `'DELETED'`, etc.).

> **Tradeoff vs. native UPDATE/DELETE CDC:** With an insert-only event log you lose native
> `delete` and `update_before`/`update_after` change types. All events look like inserts.
> The benefit is full compatibility with `table_changes` without requiring a separate CDC tool.


---
## 🧹 Cleanup
Uncomment the following to drop the tables and views used in this demo.

In [ ]:
# run_query("DROP TABLE IF EXISTS iceberg.gold.daily_sales")
# run_query("DROP TABLE IF EXISTS iceberg.silver.orders")
# run_query("DROP TABLE IF EXISTS iceberg.silver.customers")
# print("🗑️ Cleanup complete.")